# This file is concerned with the QA pairs for Blumatix documents
The QA pairs are made up of:
- 1 Reference document
- 20 curated questions
- 20 auto generated answers

The goal of this LLM as a Judge (LLMJ) application is, to test whether the auto generated answer sufficiently answer the question.<br>
For this, multi trace reasoning with majority voting and self-assesment will be used.

## LLM Setup

In [32]:
# --- setup: remove retries, unused vars, fix concurrent name, simpler mkdir ---
from datetime import datetime
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import AsyncOpenAI
import logging
import json

load_dotenv(override=True)

DOCUMENT_STORAGE = Path(os.getenv("DOCUMENT_STORAGE_QA"))
TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

client = AsyncOpenAI(api_key=API_KEY, base_url=ENDPOINT)

CONCURRENT_TASKS = 15

logging_path = Path("../QALogs")
logging_path.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    filename=f"{logging_path}/{DEPLOYMENT_NAME}_{TIMESTAMP}.log",
    filemode="a",
    format="%(asctime)s - %(levelname)s - %(message)s",
    level=logging.INFO,
    force=True,
)
for noisy in ["httpx", "openai", "azure", "urllib3"]:
    logging.getLogger(noisy).setLevel(logging.WARNING)


## Retrieval of documents

In [33]:
from pathlib import Path
import pdfplumber

def load_pdfs_for_context(folder_path: str) -> dict[str, str]:
    """
    Retrieve all .pdf files in a folder, extract text,
    and return a dict mapping filenames to their extracted text.
    """
    folder = Path(folder_path)
    pdf_files = sorted(folder.rglob("*.pdf"))
    context = {}

    for pdf_file in pdf_files:
        try:
            with pdfplumber.open(pdf_file) as pdf:
                text = "\n".join(page.extract_text() or "" for page in pdf.pages)
            context[pdf_file.name] = text.strip()
        except Exception as e:
            print(f"Error reading {pdf_file}: {e}")

    return context

pdf_contents = load_pdfs_for_context(DOCUMENT_STORAGE / "docs")
print(f"Loaded {len(pdf_contents)} PDF documents")

# Load QA pairs from DocumentQA_new.json
qa_file_path = DOCUMENT_STORAGE / "DocumentQA_new.json"
with open(qa_file_path, 'r', encoding='utf-8') as f:
    qa_data = json.load(f)

# Store QA pairs in a dictionary organized by document name
qa_pairs_by_document = {}
for item in qa_data:
    doc_name = item["document_name"]
    qa_pairs_by_document[doc_name] = item["qa_pairs"]

print(f"Loaded QA pairs for {len(qa_pairs_by_document)} documents")
print(f"Total QA pairs: {sum(len(pairs) for pairs in qa_pairs_by_document.values())}")


Loaded 10 PDF documents
Loaded QA pairs for 10 documents
Total QA pairs: 200


## LLM Call

In [53]:
# Updated _timed_request function using client.responses.create API
import time
import asyncio
from openai import BadRequestError

CONFIDENCE_CLASSES = [
    "Almost no chance",
    "Highly unlikely",
    "Chances are slight",
    "Unlikely",
    "Less than even",
    "Better than even",
    "Likely",
    "Very good chance",
    "Highly likely",
    "Almost certain"
]

def score_to_verbal(score: float) -> str:
    s = 0.0 if score is None else float(score)
    s = max(0.0, min(1.0, s))
    bounds = [0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
    idx = max(i for i, b in enumerate(bounds) if s >= b)
    return CONFIDENCE_CLASSES[idx]

async def _timed_request(model, instructions, input_text, reasoning=None, max_output_tokens=2000, text=None, user="qa-judge", store=True, timeout_sec=60):
    """Timed request function using client.responses.create API"""
    start = time.time()
    try:
        resp = await asyncio.wait_for(
            client.responses.create(
                model=model,
                instructions=instructions,
                input=input_text,
                reasoning=reasoning,
                max_output_tokens=max_output_tokens,
                text=text,
                user=user,
                store=store
            ), 
            timeout=timeout_sec
        )
        
    except (asyncio.TimeoutError, BadRequestError) as e:
        runtime = round(time.time() - start, 2)
        print(f"Request failed with error: {e}")
        return None, {
            "runtime_sec": runtime,
            "input_tokens": None,
            "cached_input_tokens": None,
            "output_tokens": None,
            "total_tokens": None,
            "timed_out": True,
        }
    except Exception as e:
        runtime = round(time.time() - start, 2)
        print(f"Request failed with error: {e}")
        print(f"Error type: {type(e)}")
        logging.error(f"Request failed: {e}")
        return None, {
            "runtime_sec": runtime,
            "input_tokens": None,
            "cached_input_tokens": None,
            "output_tokens": None,
            "total_tokens": None,
            "timed_out": False,
        }
    
    runtime = round(time.time() - start, 2)
    usage = getattr(resp, "usage", None)
    ptd = getattr(usage, "prompt_tokens_details", None) if usage else None
    cached = getattr(ptd, "cached_tokens", None) if ptd else None
    
    print(f"Request successful. Runtime: {runtime}s")
    
    return resp, {
        "runtime_sec": runtime,
        "input_tokens": getattr(usage, "input_tokens", None),
        "cached_input_tokens": cached,
        "output_tokens": getattr(usage, "output_tokens", None),
        "total_tokens": getattr(usage, "total_tokens", None),
        "timed_out": False,
    }

def _extract_reasoning_text(response):
    """Extract reasoning text from response - updated for responses.create API"""
    if response and hasattr(response, 'output_text'):
        return response.output_text
    return ""

# Updated QA validation function using the fixed request function
async def run_three_step_qa_validation(question: str, answer: str, pdf_context: str, doc_name: str):
    """
    3-step reasoning process to validate if an answer correctly answers a question given PDF context.
    Returns: True/False/Unknown, reasoning, confidence (0-1)
    """
    
    system_prompt = """You are an expert document analyst. Your task is to evaluate whether a given answer correctly and sufficiently answers a question based on the provided document context. Be thorough, accurate, and provide clear reasoning."""
    
    # Step 1 — CONTEXT ANALYSIS
    resp1, log1 = await _timed_request(
        model=DEPLOYMENT_NAME,
        instructions=system_prompt,
        input_text=(
            "Step 1 — CONTEXT ANALYSIS:\n"
            "Analyze the provided document context to understand what information is available to answer the question. "
            "Identify key facts, data points, and relevant sections. "
            "Do NOT evaluate the answer yet - just understand what the document says.\n\n"
            f"--- DOCUMENT CONTEXT START ---\n{pdf_context}\n--- DOCUMENT CONTEXT END ---\n\n"
            f"--- QUESTION ---\n{question}\n"
        ),
        reasoning={"effort": "low"},
        text={"verbosity": "low"}
    )
    
    if resp1 is None:
        return None
    step1_notes = _extract_reasoning_text(resp1)
    
    # Step 2 — ANSWER EVALUATION  
    resp2, log2 = await _timed_request(
        model=DEPLOYMENT_NAME,
        instructions=system_prompt,
        input_text=(
            "Step 2 — ANSWER EVALUATION:\n"
            "Now evaluate the provided answer against the question and document context. "
            "Check for accuracy, completeness, and whether it properly addresses the question. "
            "Consider if the answer contains incorrect information, missing key points, or irrelevant details. "
            "Do NOT provide your final judgment yet.\n\n"
            f"--- PREVIOUS CONTEXT ANALYSIS ---\n{step1_notes}\n\n"
            f"--- QUESTION ---\n{question}\n\n"
            f"--- ANSWER TO EVALUATE ---\n{answer}\n"
        ),
        reasoning={"effort": "low"},
        text={"verbosity": "low"}
    )
    
    if resp2 is None:
        return None
    step2_notes = _extract_reasoning_text(resp2)
    
    # Step 3 — FINAL JUDGMENT
    resp3, log3 = await _timed_request(
        model=DEPLOYMENT_NAME,
        instructions=system_prompt,
        input_text=(
            "Step 3 — FINAL JUDGMENT:\n"
            "Based on your analysis, provide your final evaluation. Return ONLY a single valid JSON object with this exact structure:\n"
            "{\n"
            '  "judgment": "<True|False|Unknown>",\n'
            '  "reasoning": "<detailed explanation of your decision>",\n'
            '  "confidence": <numeric value between 0.0 and 1.0>\n'
            "}\n\n"
            "Where:\n"
            "- True: The answer correctly and sufficiently answers the question based on the document\n"
            "- False: The answer is incorrect, incomplete, or doesn't properly answer the question\n"
            "- Unknown: Cannot determine due to insufficient information in the document\n"
            "- confidence: How certain you are of your judgment (0.0 = very uncertain, 1.0 = very certain)\n\n"
            f"--- CONTEXT ANALYSIS ---\n{step1_notes}\n\n"
            f"--- ANSWER EVALUATION ---\n{step2_notes}\n"
        ),
        reasoning={"effort": "low"},
        text={"verbosity": "low"}
    )
    
    if resp3 is None:
        return None
    
    try:
        result = json.loads(resp3.output_text)
        
        # Add verbal confidence description
        confidence_verbal = score_to_verbal(result.get("confidence", 0.0))
        
        # Calculate total tokens and cost (simplified)
        total_input_tokens = (log1.get("input_tokens", 0) + log2.get("input_tokens", 0) + log3.get("input_tokens", 0))
        total_output_tokens = (log1.get("output_tokens", 0) + log2.get("output_tokens", 0) + log3.get("output_tokens", 0))
        
        final_result = {
            "document_name": doc_name,
            "question": question,
            "answer": answer,
            "judgment": result.get("judgment"),
            "reasoning": result.get("reasoning"),
            "confidence_numeric": result.get("confidence", 0.0),
            "confidence_verbal": confidence_verbal,
            "step_analysis": {
                "step1_context": step1_notes,
                "step2_evaluation": step2_notes
            },
            "llm_logs": {
                "step1": log1,
                "step2": log2, 
                "step3": log3
            },
            "token_usage": {
                "total_input_tokens": total_input_tokens,
                "total_output_tokens": total_output_tokens
            }
        }
        
        return final_result
        
    except json.JSONDecodeError as e:
        logging.error(f"Failed to parse JSON response: {e}")
        return None

print("QA validation function loaded successfully!")


QA validation function loaded successfully!


In [55]:
# Example usage with loaded data
# Let's test with the first document and first QA pair
doc_name = list(qa_pairs_by_document.keys())[0]
qa_pair = qa_pairs_by_document[doc_name][1]

print(f"Testing with document: {doc_name}")
print(f"Question: {qa_pair['Question']}")

result = await run_three_step_qa_validation(
    question=qa_pair["Question"],
    answer=qa_pair["Detailed Answer"], 
    pdf_context=pdf_contents[doc_name],
    doc_name=doc_name
)

if result:
    print(f"Judgment: {result['judgment']}")
    print(f"Confidence: {result['confidence_numeric']:.2f} ({result['confidence_verbal']})")
    print(f"Reasoning: {result['reasoning']}")
else:
    print("Failed to get result")

Testing with document: Accounts Payable Processing_BLU DELTA_Nintex Platform-V2_Nov 2022_de-NBCHRIS.pdf
Question: Welche Rolle spielt künstliche Intelligenz in der Verarbeitung von Rechnungsdokumenten bei BLU DELTA?
Request successful. Runtime: 5.23s
Request successful. Runtime: 6.07s
Request successful. Runtime: 3.53s
Judgment: True
Confidence: 0.90 (Almost certain)
Reasoning: The answer accurately reflects the document: it states AI is central to BLU DELTA's document/invoice processing; it cites use of modern architectures (NLP, deep learning) for unstructured inputs (scans, PDFs); it describes extraction of header/line/footer data and semantic/positional understanding; it explains the central/shared KI service that learns from pooled document sets to improve recognition; it notes anonymized processing for privacy; and it references quality assurance (benchmarks, thousands of checks, millions of interpretations) and resulting improvements in speed, accuracy, scalability, and automati